# Practical 1 - What Is a Convolutional Neural Network Looking At?

**[Advanced Data Science and Machine Learning for Health Research](https://erasmussummerprogramme.nl/summer-programme-courses/advanced-data-science-and-machine-learning-for-health-research)**

---

## The question of this practical

> A convolutional neural network makes an accurate prediction on medical images.
> **How can we investigate what information it is actually using?**

## Learning objectives

After this notebook you should be able to:

1. describe how a CNN builds **hierarchical representations** (image -> edges -> textures -> structures -> high-level features -> prediction);
2. **inspect intermediate feature maps** of a convolutional network;
3. use an **openly available pretrained** convolutional model;
4. explain what **model attribution** does and does not measure;
5. compute and interpret **Grad-CAM** maps;
6. distinguish **predictive accuracy** from **biological plausibility**;
7. recognise **shortcut learning** in a controlled experiment;
8. connect shortcut learning to **confounding, domain shift and external validity**.

## CORE vs ADDITIONAL

> **CORE — everyone should complete**
>
> Run the guided cells, look at the figures, and discuss the questions with your neighbour.
>
> 1. Feature maps
> 2. Grad-CAM
> 3. Perturbation experiment
> 4. Shortcut-learning experiment
>
> **ADDITIONAL — optional coding**
>
> Exercises 1–4. These are extra practice if you have time and want to try Python. They are
> not required, and the scientific message of the practical does not depend on writing code.

In a 75–90 minute session, finish the CORE path carefully. Additional coding is only for
groups who want extra practice.

## How this notebook is organised

| Section label | What it asks of you |
|---|---|
| **Concept** | Short theory. Read it. |
| **Health-research perspective** | Why this matters clinically or epidemiologically. |
| **Think before running** | Predict the outcome with your neighbour *before* executing the next cell. |
| **Interpretation** | Think about the scientific questions and discuss them with your neighbour. |
| **Additional exercise** | Optional Python. Skip unless you want extra practice. |

Whenever this notebook asks a question, think it through and discuss it with the person next
to you. You do not need to write answers down.

The notebook is guided: you run prepared cells and discuss. Coding is extra.

## Computational profile

* Runs end to end on a **free Colab CPU runtime**. A GPU is used automatically if present but is never required.
* Downloads: **~21 MB** dataset + **~45 MB** pretrained ImageNet weights.
* Two short training runs, each **~1-2 minutes on CPU**. Trained weights are cached, so re-running a cell does not retrain.
* Peak RAM well below 4 GB.
* Expected duration: **80-90 minutes**.

---

## Dataset and reproducibility

| Item | Value |
|---|---|
| **Dataset** | PneumoniaMNIST, 64x64 version (from the MedMNIST v2 / MedMNIST+ collection) |
| **Biomedical modality** | Paediatric chest radiography (frontal chest X-ray) |
| **Source (download)** | Zenodo record [10519652](https://zenodo.org/records/10519652), file `pneumoniamnist_64.npz` (20.6 MB), official MedMNIST distribution |
| **Underlying clinical data** | 5,856 paediatric chest X-rays (ages 1-5) collected during routine care at Guangzhou Women and Children's Medical Center, China |
| **Labels** | Binary: `0 = normal`, `1 = pneumonia` (expert-graded in the source study) |
| **Official splits** | 4,708 train / 524 validation / 624 test (patient-level split done by the source authors) |
| **Sample size used here** | Class-balanced subsets drawn with a fixed seed: **2,400 train** (1,200 per class), **~270 validation**, **400 test** (200 per class) |
| **Preprocessing by MedMNIST authors** | Grayscale, centre-cropped, resized to 64x64, stored as `uint8` |
| **Preprocessing in this notebook** | Scale to [0, 1]; for our small CNN normalise to [-1, 1]; for ImageNet ResNet-18 replicate the grey channel to 3 channels and apply ImageNet mean/std |
| **License / usage** | MedMNIST data: **CC BY 4.0**; MedMNIST code: Apache-2.0. Source images from Kermany et al. released under CC BY 4.0. **Not for clinical use.** |
| **Pretrained model** | `torchvision` ResNet-18, ImageNet-1k weights (`IMAGENET1K_V1`), BSD-3-Clause library license. Trained on **natural photographs, not medical images** |

### Citations to use if you reuse this data

```
Yang, J., Shi, R., Wei, D., Liu, Z., Zhao, L., Ke, B., Pfister, H., Ni, B. (2023).
MedMNIST v2 - A large-scale lightweight benchmark for 2D and 3D biomedical image
classification. Scientific Data, 10(1), 41.

Yang, J., Shi, R., Ni, B. (2021). MedMNIST Classification Decathlon: A Lightweight
AutoML Benchmark for Medical Image Analysis. IEEE ISBI 2021, 191-195.

Kermany, D. S., Goldbaum, M., Cai, W., et al. (2018). Identifying Medical Diagnoses
and Treatable Diseases by Image-Based Deep Learning. Cell, 172(5), 1122-1131.

He, K., Zhang, X., Ren, S., Sun, J. (2016). Deep Residual Learning for Image
Recognition. CVPR 2016. (ResNet-18 architecture / ImageNet weights)
```

> ### Honesty statement about the data
>
> This notebook contains **no real clinical metadata**: no age, sex, scanner, hospital or outcome
> variables beyond the published diagnostic label. Later in the notebook we deliberately
> **create synthetic image manipulations** (a marker square, occlusions, contrast changes).
> Every such manipulation is explicitly labelled as **simulated**, and must never be
> presented as a property of the original data.
>
> The class-balanced subsets used here (50% pneumonia) are a *teaching convenience*.
> They do not reflect the prevalence of pneumonia in any real population or clinic.

---

# 0. Setup

Run the next cell once. It imports everything, fixes random seeds, detects CPU/GPU, and defines a
download helper with MD5 verification. **Nothing needs to be installed** in a standard Colab
runtime (PyTorch, torchvision, scikit-learn and matplotlib are preinstalled), and you do not need
Google Drive, credentials or manual uploads.

In [ ]:
# =====================================================================
# SETUP CELL - run this first
# =====================================================================
import hashlib
import os
import random
import sys
import time
import urllib.request
from contextlib import contextmanager

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                            confusion_matrix, roc_auc_score)

# ---------- reproducibility ----------
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
RNG = np.random.default_rng(SEED)          # use this generator for all sampling

# ---------- device ----------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() or 1))

# ---------- folders ----------
DATA_DIR, WEIGHT_DIR = "medmnist_data", "cached_weights"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(WEIGHT_DIR, exist_ok=True)

# ---------- instructor weights (panic button for class) ----------
# Class default: TRAIN_* = False loads committed checkpoints from ./weights/ or GitHub.
# Set TRAIN_* = True only if you want to reproduce training (~1-4 min on CPU).
WEIGHTS_REPO = "https://github.com/roshchupkin/Advanced_ML_2026/raw/main/weights"
INSTRUCTOR_WEIGHT_DIR = "weights"
os.makedirs(INSTRUCTOR_WEIGHT_DIR, exist_ok=True)


def resolve_weight(filename):
    """Return a local path to `filename`, downloading from GitHub if needed."""
    local = os.path.join(INSTRUCTOR_WEIGHT_DIR, filename)
    if os.path.exists(local):
        return local
    url = f"{WEIGHTS_REPO}/{filename}"
    print(f"instructor weight not found locally - downloading {filename} ...")
    download(url, local)
    return local


def load_or_train(model, filename, train_flag, fit_fn, label):
    """Load instructor weights unless train_flag is True."""
    cache_path = os.path.join(WEIGHT_DIR, filename)
    if train_flag:
        print(f"TRAIN flag is True - training {label} ...")
        history = fit_fn()
        torch.save(model.state_dict(), cache_path)
        print(f"saved trained weights to {cache_path}")
        return history
    path = resolve_weight(filename)
    try:
        state = torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)
    print(f"loaded instructor weights from {path}  (set TRAIN flag to True to retrain)")
    return None


def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def download(url, dest, expected_md5=None):
    # Download `url` to `dest` unless a valid copy already exists.
    if os.path.exists(dest):
        if expected_md5 is None or md5sum(dest) == expected_md5:
            print(f"already present: {dest} ({os.path.getsize(dest)/1e6:.1f} MB)")
            return dest
        print("existing file failed the MD5 check - downloading again")

    def hook(blocks, block_size, total):
        if total > 0:
            pct = min(100.0, 100.0 * blocks * block_size / total)
            print(f"\rdownloading {os.path.basename(dest)}: {pct:5.1f}%", end="")

    t0 = time.time()
    urllib.request.urlretrieve(url, dest, reporthook=hook)
    print(f"\rdownloaded {os.path.basename(dest)}: "
          f"{os.path.getsize(dest)/1e6:.1f} MB in {time.time()-t0:.0f} s")
    if expected_md5 is not None:
        got = md5sum(dest)
        if got != expected_md5:
            print(f"WARNING: MD5 mismatch (expected {expected_md5}, got {got}). "
                  "The file may be corrupted; try re-running this cell.")
        else:
            print("MD5 verified.")
    return dest


@contextmanager
def timed(label):
    t0 = time.time()
    yield
    print(f"[{label}] took {time.time() - t0:.1f} s")


def report_environment():
    print("python           :", sys.version.split()[0])
    print("torch            :", torch.__version__)
    print("torchvision      :", torchvision.__version__)
    print("device           :", DEVICE.type.upper(),
          f"({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else "")
    print("cpu threads      :", torch.get_num_threads())
    try:
        import psutil
        vm = psutil.virtual_memory()
        print("RAM total / free :", f"{vm.total/1e9:.1f} GB / {vm.available/1e9:.1f} GB")
    except Exception:
        print("RAM              : (psutil unavailable)")


plt.rcParams.update({"figure.dpi": 110, "axes.grid": False,
                     "image.interpolation": "nearest", "font.size": 9})
TRAIN_CLEAN = False      # set True to retrain the clean SmallCNN (~1-2 min CPU)
TRAIN_SHORTCUT = False   # set True to retrain the shortcut SmallCNN (~1-2 min CPU)

report_environment()
print(f"TRAIN_CLEAN={TRAIN_CLEAN}, TRAIN_SHORTCUT={TRAIN_SHORTCUT}")
print("\nSetup complete.")

In [ ]:
# =====================================================================
# Download PneumoniaMNIST (64x64) from the official Zenodo record
# =====================================================================
PNEUMONIA_URL = "https://zenodo.org/records/10519652/files/pneumoniamnist_64.npz?download=1"
PNEUMONIA_MD5 = "8f4eceb4ccffa70c672198ea285246c6"
npz_path = os.path.join(DATA_DIR, "pneumoniamnist_64.npz")

# If Zenodo is unreachable from your runtime, the official package is a drop-in
# alternative:  !pip install medmnist   then
#   from medmnist import PneumoniaMNIST; PneumoniaMNIST(split="train", size=64, download=True)
download(PNEUMONIA_URL, npz_path, PNEUMONIA_MD5)

blob = np.load(npz_path)
print("\narrays inside the npz:", sorted(blob.files))

X_train_all = blob["train_images"]
y_train_all = blob["train_labels"].ravel().astype(np.int64)
X_val_all = blob["val_images"]
y_val_all = blob["val_labels"].ravel().astype(np.int64)
X_test_all = blob["test_images"]
y_test_all = blob["test_labels"].ravel().astype(np.int64)

CLASS_NAMES = ["normal", "pneumonia"]

print(f"\ntrain images {X_train_all.shape} dtype={X_train_all.dtype} "
      f"range=[{X_train_all.min()}, {X_train_all.max()}]")
for name, y in [("train", y_train_all), ("val", y_val_all), ("test", y_test_all)]:
    counts = np.bincount(y, minlength=2)
    print(f"{name:5s}: n={len(y):5d}  normal={counts[0]:4d}  pneumonia={counts[1]:4d} "
          f"(pneumonia = {100*counts[1]/len(y):.1f}%)")
print(f"\nmemory held by all images: "
      f"{(X_train_all.nbytes + X_val_all.nbytes + X_test_all.nbytes)/1e6:.1f} MB (uint8)")

---

# Part 1 - Very short recap: what a convolution actually computes

### Concept

A 2D **convolution** slides a small weight matrix (the **kernel** or filter) over the image and
computes a weighted sum at every position:

$$ S(i,j) \;=\; (I * K)(i,j) \;=\; \sum_{u}\sum_{v} I(i+u,\; j+v)\; K(u,v) $$

* **Kernel**: a small matrix of learnable weights, typically 3x3. A CNN layer with `C_out` kernels has `C_out * C_in * k * k` weights.
* **Feature map**: the output image produced by one kernel. It answers "where in the image does this pattern occur?"
* **Channels**: the stack of feature maps at a given layer. Channel count is *representational* width, not colour.
* **Weight sharing**: the same kernel is applied everywhere, which is why CNNs are translation-equivariant and need far fewer parameters than a dense network.
* **Receptive field**: the region of the *input* image that influences one unit deep in the network. It grows with depth, kernel size and stride.

The key consequence of stacking convolutions with downsampling is a **hierarchy**:

```
image  ->  edges  ->  textures  ->  local structures  ->  high-level representation  ->  prediction
 64x64      64x64        32x32            16x16                  4x4 or 1x1              2 logits
(spatial detail decreases, semantic abstraction increases)
```

### Think before running

The next cell convolves one chest X-ray with **hand-designed** kernels (no learning involved).

1. Which of these kernels do you expect to *destroy* diagnostic information: a 3x3 blur, or a Sobel edge detector?
2. The output of a 3x3 convolution with `padding=1` has the same height and width as the input. What would happen to the output size with `padding=0`?

In [ ]:
# Hand-designed kernels applied to one radiograph: no training, pure linear filtering
demo_img = X_test_all[3]                                     # (64, 64) uint8
x_demo = torch.from_numpy(demo_img).float().div(255.).view(1, 1, 64, 64)

kernels = {
    "identity":  [[0, 0, 0], [0, 1, 0], [0, 0, 0]],
    "blur 3x3":  [[1/9, 1/9, 1/9], [1/9, 1/9, 1/9], [1/9, 1/9, 1/9]],
    "Sobel x":   [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "Sobel y":   [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
    "Laplacian": [[0, -1, 0], [-1, 4, -1], [0, -1, 0]],
}

fig, axes = plt.subplots(1, len(kernels) + 1, figsize=(2.0 * (len(kernels) + 1), 2.4))
axes[0].imshow(demo_img, cmap="gray")
axes[0].set_title(f"input\n{demo_img.shape}", fontsize=8)
for ax, (name, k) in zip(axes[1:], kernels.items()):
    w = torch.tensor(k, dtype=torch.float32).view(1, 1, 3, 3)
    out = F.conv2d(x_demo, w, padding=1)[0, 0].numpy()
    ax.imshow(out, cmap="gray")
    ax.set_title(f"{name}\n{out.shape}", fontsize=8)
for ax in axes:
    ax.axis("off")
fig.suptitle("One image, five 3x3 kernels - each output is a feature map", fontsize=10)
plt.tight_layout()
plt.show()

# same kernel, no padding: the feature map shrinks by (k - 1) pixels per axis
no_pad = F.conv2d(x_demo, torch.ones(1, 1, 3, 3) / 9.0, padding=0)
print("with padding=1 :", tuple(F.conv2d(x_demo, torch.ones(1, 1, 3, 3) / 9.0, padding=1).shape))
print("with padding=0 :", tuple(no_pad.shape))

### Concept - receptive fields grow surprisingly fast

For a stack of layers with kernel sizes $k_\ell$ and strides $s_\ell$, the receptive field is

$$ r_L = 1 + \sum_{\ell=1}^{L} (k_\ell - 1) \prod_{i<\ell} s_i $$

The next cell computes this for ResNet-18, the pretrained network we will use.

### Health-research perspective

Receptive field size determines **what kind of evidence a model can use at all**. A unit whose
receptive field covers the whole thorax can respond to *global* properties: overall exposure,
patient positioning, the presence of a text annotation burned into the corner of the image, or
the scanner's noise signature. These are exactly the features that differ between hospitals,
and they are a frequent cause of models that validate beautifully and then fail on external data.

In [ ]:
# Receptive field arithmetic for ResNet-18 (input 64x64)
stages = ([("conv1 7x7 s2", 7, 2), ("maxpool 3x3 s2", 3, 2)]
          + [(f"layer1.conv{i} 3x3 s1", 3, 1) for i in range(1, 5)]
          + [("layer2.conv1 3x3 s2", 3, 2)] + [(f"layer2.conv{i} 3x3 s1", 3, 1) for i in range(2, 5)]
          + [("layer3.conv1 3x3 s2", 3, 2)] + [(f"layer3.conv{i} 3x3 s1", 3, 1) for i in range(2, 5)]
          + [("layer4.conv1 3x3 s2", 3, 2)] + [(f"layer4.conv{i} 3x3 s1", 3, 1) for i in range(2, 5)])

rf, jump = 1, 1
print(f"{'layer':24s} {'receptive field':>16s} {'effective stride':>17s}")
for name, k, s in stages:
    rf = rf + (k - 1) * jump
    jump = jump * s
    print(f"{name:24s} {rf:>13d} px {jump:>15d} px")
print("\nInput image is 64 x 64 px.")
print("Deep units therefore see the ENTIRE image: their 'evidence' can be global, "
      "not local anatomy.")

---

# Part 2 - Load the biomedical images

### Health-research perspective

Before any modelling, note what this dataset *is*: paediatric chest radiographs from **one
hospital**, labelled during routine care, in patients aged 1-5. Any model trained here inherits

* the **spectrum of disease** of that centre (referral patterns, severity mix),
* the **acquisition protocol** of that centre (device, exposure, positioning, collimation),
* the **labelling process** of that study (radiologist grading with a specific definition of pneumonia).

For teaching we draw **class-balanced** subsets with a fixed seed. This is a deliberate
distortion of prevalence that makes accuracy easier to read, and it means the absolute numbers
in this notebook are not transferable to any clinical setting.

In [ ]:
def stratified_subset(y, n_per_class, rng):
    # Indices of a class-balanced random subset; the fixed generator keeps it reproducible.
    picked = []
    for c in np.unique(y):
        candidates = np.flatnonzero(y == c)
        take = min(n_per_class, len(candidates))
        picked.append(rng.choice(candidates, size=take, replace=False))
    out = np.concatenate(picked)
    rng.shuffle(out)
    return out


tr_idx = stratified_subset(y_train_all, 1200, RNG)
va_idx = stratified_subset(y_val_all, 135, RNG)
te_idx = stratified_subset(y_test_all, 200, RNG)

X_tr, y_tr = X_train_all[tr_idx], y_train_all[tr_idx]
X_va, y_va = X_val_all[va_idx], y_val_all[va_idx]
X_te, y_te = X_test_all[te_idx], y_test_all[te_idx]

for name, X, y in [("train", X_tr, y_tr), ("val", X_va, y_va), ("test", X_te, y_te)]:
    counts = np.bincount(y, minlength=2)
    print(f"{name:5s}: X {str(X.shape):16s} normal={counts[0]:4d} pneumonia={counts[1]:4d} "
          f"({X.nbytes/1e6:.1f} MB)")

# a look at the raw material
fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))
for row, cls in enumerate([0, 1]):
    ids = np.flatnonzero(y_tr == cls)[:6]
    for col, i in enumerate(ids):
        axes[row, col].imshow(X_tr[i], cmap="gray", vmin=0, vmax=255)
        axes[row, col].axis("off")
        axes[row, col].set_title(CLASS_NAMES[cls], fontsize=8)
fig.suptitle("PneumoniaMNIST 64x64 - top: normal, bottom: pneumonia", fontsize=10)
plt.tight_layout()
plt.show()

### Interpretation

Look carefully at the twelve images above. Think about each question and discuss it with your neighbour:

1. Can *you* separate the two classes by eye? Which visual cues are you using?
2. Which non-diagnostic differences can you already spot (framing, contrast, how much of the
   abdomen is included, rotation)?
3. If a model reached 95% accuracy on this dataset, name two explanations other than
   "it detects consolidation".

---

# Part 3 - Inspect the models and their predictions

We will interrogate **two** convolutional networks, because they fail in interestingly different ways.

| | **Model A: ResNet-18** | **Model B: SmallCNN** |
|---|---|---|
| Weights | pretrained on **ImageNet** (natural photographs) | trained here on PneumoniaMNIST |
| Parameters | ~11.7 million | ~24 thousand |
| Role in this notebook | architecture and feature-map inspection, transfer-learning reference | the classifier we will explain with Grad-CAM |
| Training cost | none (weights downloaded) | ~1-2 min on CPU |

### Concept - why such a small model?

The pedagogical goal is *interpretation*, not scale. A 24k-parameter CNN reaches high accuracy on
64x64 PneumoniaMNIST, trains in about a minute on a CPU, and gives Grad-CAM maps at 16x16
resolution. A ResNet-18 at this input size collapses to a 2x2 grid in its last convolutional
block, which is too coarse to localise anything (we will see this explicitly).

### Think before running

1. ResNet-18 has ~11.7M parameters. How many of them are in the **final fully connected layer**
   alone (512 features -> 1000 classes)? Is most of the capacity convolutional or dense?
2. Model A was trained on photographs of dogs, cars and mushrooms. What do you expect it to
   predict for a chest radiograph?

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def prep_gray(x_u8, device=None):
    # (N, H, W) or (H, W) uint8  ->  (N, 1, H, W) float tensor scaled to [-1, 1]
    if x_u8.ndim == 2:
        x_u8 = x_u8[None, ...]
    t = torch.from_numpy(np.ascontiguousarray(x_u8)).float().div_(255.0)
    t = (t - 0.5) / 0.5
    return t.unsqueeze(1).to(device or DEVICE)


def prep_rgb_imagenet(x_u8, device=None):
    # grayscale -> 3 identical channels + ImageNet normalisation (what ResNet-18 expects)
    if x_u8.ndim == 2:
        x_u8 = x_u8[None, ...]
    t = torch.from_numpy(np.ascontiguousarray(x_u8)).float().div_(255.0)
    t = t.unsqueeze(1).repeat(1, 3, 1, 1)
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return ((t - mean) / std).to(device or DEVICE)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def build_pretrained_resnet18():
    # ~45 MB of ImageNet weights, cached by torch under ~/.cache/torch
    categories = None
    try:
        from torchvision.models import ResNet18_Weights, resnet18
        weights = ResNet18_Weights.IMAGENET1K_V1
        model = resnet18(weights=weights)
        categories = list(weights.meta.get("categories", []))
    except Exception as exc:
        print("New torchvision weights API unavailable, using the legacy one:", exc)
        model = torchvision.models.resnet18(pretrained=True)
    return model.eval().to(DEVICE), categories


with timed("load pretrained ResNet-18"):
    resnet, imagenet_categories = build_pretrained_resnet18()

total, trainable = count_parameters(resnet)
fc_params = resnet.fc.weight.numel() + resnet.fc.bias.numel()
print(f"\nResNet-18: {total:,} parameters ({trainable:,} trainable)")
print(f"final fully connected layer alone: {fc_params:,} parameters "
      f"({100*fc_params/total:.1f}% of the model)")
print("top-level blocks:", [name for name, _ in resnet.named_children()])

In [ ]:
def trace_shapes(model, x, layer_names):
    # Run one batch and record the output shape of each named submodule.
    shapes, handles = {}, []
    modules = dict(model.named_modules())
    for name in layer_names:
        handles.append(modules[name].register_forward_hook(
            lambda mod, inp, out, name=name: shapes.__setitem__(name, tuple(out.shape))))
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()
    model.train(was_training)
    return shapes


x_probe = prep_rgb_imagenet(X_te[:4])
print("input tensor:", tuple(x_probe.shape), " = (batch, channels, height, width)\n")

resnet_stages = ["conv1", "maxpool", "layer1", "layer2", "layer3", "layer4", "avgpool", "fc"]
shapes = trace_shapes(resnet, x_probe, resnet_stages)
print(f"{'stage':10s} {'output shape':22s} {'channels':>9s} {'spatial':>9s} {'values/image':>13s}")
for name in resnet_stages:
    s = shapes[name]
    channels = s[1] if len(s) > 1 else 1
    spatial = f"{s[2]}x{s[3]}" if len(s) == 4 else "-"
    print(f"{name:10s} {str(s):22s} {channels:>9d} {spatial:>9s} "
          f"{int(np.prod(s[1:])):>13,d}")
print("\nFor a 64x64 input the spatial grid shrinks 32 -> 16 -> 16 -> 8 -> 4 -> 2 through "
      "conv1, maxpool, layer1 ... layer4,")
print("while the number of channels grows 64 -> 512.")
print("That is the hierarchy: less 'where', more 'what'.")

In [ ]:
# What does an ImageNet model think a chest X-ray is? (sanity demonstration of domain mismatch)
if imagenet_categories:
    with torch.no_grad():
        probs_imnet = torch.softmax(resnet(prep_rgb_imagenet(X_te[:4])), dim=1).cpu()
    fig, axes = plt.subplots(1, 4, figsize=(9, 2.9))
    for i in range(4):
        axes[i].imshow(X_te[i], cmap="gray")
        axes[i].axis("off")
        p, idx = probs_imnet[i].topk(3)
        axes[i].set_title("\n".join(f"{imagenet_categories[j]}: {100*v:.1f}%"
                                    for v, j in zip(p.tolist(), idx.tolist())), fontsize=7)
    fig.suptitle("ResNet-18 (ImageNet) top-3 labels for chest radiographs", fontsize=10)
    plt.tight_layout()
    plt.show()
    print("The labels are meaningless here: the OUTPUT layer is useless for radiographs.")
    print("The intermediate FEATURES (edges, textures) are still generic and often transferable.")
else:
    print("ImageNet category names are unavailable in this torchvision version - skipping demo.")

### Health-research perspective

This is transfer learning in a nutshell: the *head* of a pretrained network is domain specific
and disposable, while the early convolutional features (edges, textures, gradients) are largely
generic. Most published medical imaging models start from ImageNet weights for exactly this
reason - and inherit ImageNet's preprocessing conventions, 3-channel input assumption and
texture bias along with it.

---

### Concept - Model B, a deliberately small CNN

```
input 1x64x64
  -> block1: Conv 3x3 (1->16)  + BatchNorm + ReLU + MaxPool2   ->  16x32x32
  -> block2: Conv 3x3 (16->32) + BatchNorm + ReLU + MaxPool2   ->  32x16x16
  -> block3: Conv 3x3 (32->64) + BatchNorm + ReLU              ->  64x16x16   <- Grad-CAM target
  -> head:   GlobalAvgPool + Dropout + Linear(64 -> 2)         ->  2 logits
```

Global average pooling means the final decision is a **weighted sum of channel activations**,
which is precisely the structure Grad-CAM exploits.

### Think before running

Before training: this model has 24k parameters and sees 2,400 images. Do you expect
(a) severe overfitting, (b) reasonable generalisation, or (c) underfitting? Think about your
guess and the validation accuracy you expect, and discuss it with your neighbour.

In [ ]:
class SmallCNN(nn.Module):
    # Deliberately small CNN. block3 has no pooling, so its 16x16 grid is a usable
    # spatial resolution for Grad-CAM on 64x64 inputs.
    def __init__(self, n_classes=2, in_channels=1, width=16):
        super().__init__()
        w1, w2, w3 = width, 2 * width, 4 * width
        self.block1 = nn.Sequential(nn.Conv2d(in_channels, w1, 3, padding=1),
                                    nn.BatchNorm2d(w1), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(w1, w2, 3, padding=1),
                                    nn.BatchNorm2d(w2), nn.ReLU(), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(nn.Conv2d(w2, w3, 3, padding=1),
                                    nn.BatchNorm2d(w3), nn.ReLU())
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.25), nn.Linear(w3, n_classes))

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.head(x)


cnn = SmallCNN().to(DEVICE)
print(cnn)
total_b, _ = count_parameters(cnn)
print(f"\nSmallCNN parameters: {total_b:,}  "
      f"(ResNet-18 is {total/total_b:.0f}x larger)")
for name, shape in trace_shapes(cnn, prep_gray(X_te[:4]), ["block1", "block2", "block3"]).items():
    print(f"{name}: {shape}")

In [ ]:
def predict_probs(model, X_u8, batch=128, prep=prep_gray):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in range(0, len(X_u8), batch):
            logits = model(prep(X_u8[start:start + batch]))
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(chunks)


def train_classifier(model, X_u8, y, X_val=None, y_val=None, epochs=6, batch=64,
                     lr=1e-3, seed=SEED, prep=prep_gray):
    model.to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {"epoch": [], "train_loss": [], "val_acc": []}
    n = len(X_u8)
    for epoch in range(1, epochs + 1):
        model.train()
        order = np.random.default_rng(seed + epoch).permutation(n)
        running = 0.0
        for start in range(0, n, batch):
            idx = order[start:start + batch]
            xb, yb = prep(X_u8[idx]), torch.from_numpy(y[idx]).to(DEVICE)
            optimiser.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimiser.step()
            running += loss.item() * len(idx)
        val_acc = float("nan")
        if X_val is not None:
            val_acc = accuracy_score(y_val, predict_probs(model, X_val, prep=prep).argmax(1))
        history["epoch"].append(epoch)
        history["train_loss"].append(running / n)
        history["val_acc"].append(val_acc)
        print(f"epoch {epoch}/{epochs}   train loss {running / n:.4f}   val acc {val_acc:.3f}")
    return history


with timed("obtain SmallCNN weights (clean images)"):
    history_clean = load_or_train(
        cnn, "cnn_pneumoniamnist.pt", TRAIN_CLEAN,
        lambda: train_classifier(cnn, X_tr, y_tr, X_va, y_va, epochs=6, batch=64, lr=1e-3),
        "clean SmallCNN")

if history_clean is not None:
    fig, ax = plt.subplots(1, 2, figsize=(7.5, 2.6))
    ax[0].plot(history_clean["epoch"], history_clean["train_loss"], marker="o")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training loss"); ax[0].set_title("training loss")
    ax[1].plot(history_clean["epoch"], history_clean["val_acc"], marker="o", color="tab:green")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("validation accuracy")
    ax[1].set_ylim(0.5, 1.0); ax[1].set_title("validation accuracy")
    plt.tight_layout(); plt.show()

In [ ]:
# Test-set performance of Model B (400 balanced images the model has never seen)
probs_te = predict_probs(cnn, X_te)
pred_te = probs_te.argmax(1)

print(f"accuracy          {accuracy_score(y_te, pred_te):.3f}")
print(f"balanced accuracy {balanced_accuracy_score(y_te, pred_te):.3f}")
print(f"ROC-AUC           {roc_auc_score(y_te, probs_te[:, 1]):.3f}")

cm = confusion_matrix(y_te, pred_te)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, cm[i, j], ha="center", va="center",
                   color="white" if cm[i, j] > cm.max() / 2 else "black")
ax[0].set_xticks([0, 1], CLASS_NAMES); ax[0].set_yticks([0, 1], CLASS_NAMES)
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true"); ax[0].set_title("confusion matrix")
for cls, colour in [(0, "tab:blue"), (1, "tab:red")]:
    ax[1].hist(probs_te[y_te == cls, 1], bins=20, alpha=0.6, label=CLASS_NAMES[cls], color=colour)
ax[1].set_xlabel("predicted P(pneumonia)"); ax[1].set_ylabel("count")
ax[1].legend(); ax[1].set_title("probability distribution by true class")
plt.tight_layout(); plt.show()

In [ ]:
# Individual predictions: accuracy is an average, individual images are what clinicians see
show = np.concatenate([np.flatnonzero(y_te == 0)[:4], np.flatnonzero(y_te == 1)[:4]])
fig, axes = plt.subplots(2, 4, figsize=(9, 5))
for ax, i in zip(axes.ravel(), show):
    ax.imshow(X_te[i], cmap="gray")
    ax.axis("off")
    p = probs_te[i, 1]
    correct = (p > 0.5) == bool(y_te[i])
    ax.set_title(f"true: {CLASS_NAMES[y_te[i]]}\nP(pneumonia) = {p:.2f}",
                 fontsize=8, color="darkgreen" if correct else "firebrick")
fig.suptitle("Model B predictions (green title = correct, red = wrong)", fontsize=10)
plt.tight_layout(); plt.show()

---

# Part 4 - Feature maps: what is represented at each depth?

### Concept

A **feature map** is the response of one kernel across spatial positions. Reading feature maps is
the most direct way to see the hierarchy:

* **Early layers** respond to local intensity changes: edges, gradients, bright/dark transitions. Their maps still look like the radiograph.
* **Middle layers** combine edges into textures and shapes; the maps become sparser and less recognisable.
* **Deep layers** encode abstract, class-relevant configurations on a very coarse grid; individual maps are usually uninterpretable by eye.

### Think before running

1. What information do you expect the **first** convolutional layer to preserve?
2. At which depth do you expect the feature maps to stop looking like a chest?
3. Model B's `block3` output is 64 channels of 16x16. Do you expect one specific channel to
   correspond to "consolidation"?

In [ ]:
def get_activations(model, x, layer_names):
    # Forward hooks capture the output tensors of the named submodules.
    acts, handles = {}, []
    modules = dict(model.named_modules())
    for name in layer_names:
        handles.append(modules[name].register_forward_hook(
            lambda mod, inp, out, name=name: acts.__setitem__(name, out.detach().cpu())))
    model.eval()
    with torch.no_grad():
        model(x)
    for h in handles:
        h.remove()
    return acts


def show_feature_maps(activation, title, n_channels=8, sample=0):
    a = activation[sample]                                   # (C, H, W)
    k = min(n_channels, a.shape[0])
    fig, axes = plt.subplots(1, k, figsize=(1.35 * k, 1.9))
    axes = np.atleast_1d(axes)
    for j in range(k):
        axes[j].imshow(a[j].numpy(), cmap="viridis")
        axes[j].set_title(f"ch {j}", fontsize=7)
        axes[j].axis("off")
    fig.suptitle(f"{title}   tensor per image: {tuple(a.shape)}", fontsize=9)
    plt.tight_layout(); plt.show()


img_idx = int(np.flatnonzero(y_te == 1)[0])                  # a pneumonia case
x_one_gray = prep_gray(X_te[img_idx])
x_one_rgb = prep_rgb_imagenet(X_te[img_idx])

plt.figure(figsize=(2.2, 2.4))
plt.imshow(X_te[img_idx], cmap="gray"); plt.axis("off")
plt.title(f"the image we probe\ntrue: {CLASS_NAMES[y_te[img_idx]]}, "
          f"P(pneumonia) = {probs_te[img_idx, 1]:.2f}", fontsize=8)
plt.show()

acts_b = get_activations(cnn, x_one_gray, ["block1", "block2", "block3"])
show_feature_maps(acts_b["block1"], "Model B - block1 (early)")
show_feature_maps(acts_b["block2"], "Model B - block2 (middle)")
show_feature_maps(acts_b["block3"], "Model B - block3 (deep, Grad-CAM target)")

In [ ]:
# Same probe image through the ImageNet-pretrained ResNet-18 (never trained on X-rays)
acts_a = get_activations(resnet, x_one_rgb, ["conv1", "layer1", "layer2", "layer4"])
show_feature_maps(acts_a["conv1"], "Model A - conv1 (ImageNet, early)")
show_feature_maps(acts_a["layer2"], "Model A - layer2 (ImageNet, middle)")
show_feature_maps(acts_a["layer4"], "Model A - layer4 (ImageNet, deep: 2x2 only!)")

# Averaging over channels summarises "how much total activation" per position
fig, axes = plt.subplots(1, 5, figsize=(11, 2.4))
axes[0].imshow(X_te[img_idx], cmap="gray"); axes[0].set_title("input 64x64", fontsize=8)
for ax, key in zip(axes[1:], ["conv1", "layer1", "layer2", "layer4"]):
    m = acts_a[key][0].mean(0).numpy()
    ax.imshow(m, cmap="magma")
    ax.set_title(f"{key}\nmean over {acts_a[key].shape[1]} ch, {m.shape[0]}x{m.shape[1]}",
                 fontsize=8)
for ax in axes:
    ax.axis("off")
fig.suptitle("Channel-averaged activation with increasing depth (Model A)", fontsize=10)
plt.tight_layout(); plt.show()
print("Spatial resolution at layer4 is 2x2 for a 64x64 input: any attribution map computed "
      "there can only say 'top-left-ish'.")

### Interpretation

Think about each question and discuss it with your neighbour:

1. **How do the representations change with depth** in Model B? Refer to both spatial resolution
   and visual recognisability.
2. Model A's early features look similar to Model B's even though Model A never saw a
   radiograph. Why?
3. **Can individual filters safely be assigned a biological meaning** such as "this channel
   detects consolidation"? Give at least two reasons for scepticism.
4. Suppose one deep channel activates strongly over the right lower lobe in most pneumonia cases.
   **Why might this apparently anatomical activation be misleading?** Consider what else is
   systematically located in that part of the image.

### ADDITIONAL Exercise 1 - compare early and late feature maps (optional coding)

> You do not need to write Python to complete this practical. Skip this cell unless you want extra practice.

Complete the code cell below so that it:

1. picks one **normal** and one **pneumonia** test image;
2. extracts `block1` and `block3` activations for both;
3. for each layer, plots the **channel-averaged** map of both images side by side;
4. prints the **mean absolute difference** between the two images' channel-averaged maps for each layer.

Then discuss with your neighbour: at which depth do the two classes differ more, in relative terms? Does that match
your expectation from Part 1?

In [ ]:
# ===== EXERCISE 1 =====
i_normal = int(np.flatnonzero(y_te == 0)[0])
i_pneumo = int(np.flatnonzero(y_te == 1)[0])

# TODO 1: get activations for both images.
#   hint: prep_gray accepts a stack, so X_te[[i_normal, i_pneumo]] gives a batch of 2
# acts = get_activations(cnn, prep_gray(...), ["block1", "block3"])

# TODO 2: for each layer, compute the channel-mean map of sample 0 and sample 1
#   hint: acts["block1"][0].mean(0).numpy()

# TODO 3: plot the four maps (2 layers x 2 images) and print the mean absolute difference
#   hint: np.abs(map_normal - map_pneumonia).mean()

# TODO 4: normalise the difference by the mean activation of that layer so the two layers
#         are comparable, then discuss your answer with your neighbour.

---

# Part 5 - Grad-CAM: which regions influenced this prediction?

### Concept

Grad-CAM (Selvaraju et al., ICCV 2017) combines two sources of information:

* **activations** $A^k \in \mathbb{R}^{H\times W}$ of channel $k$ in a chosen convolutional layer tell us *where* a learned feature is present;
* **gradients** $\partial y^c / \partial A^k$ tell us *how strongly* that feature pushes the score $y^c$ of the class we selected.

Grad-CAM averages the gradient over space to get one importance weight per channel,

$$ \alpha_k^c \;=\; \frac{1}{HW}\sum_{i}\sum_{j} \frac{\partial y^c}{\partial A^k_{ij}}, $$

then forms a weighted sum of the activation maps and keeps only positive evidence:

$$ L^c_{\text{Grad-CAM}} \;=\; \mathrm{ReLU}\!\left( \sum_k \alpha_k^c A^k \right). $$

The result is one coarse map (16x16 for our `block3`) which is upsampled to image size for display.

**What it is:** a first-order, class-specific summary of *which spatial positions in one layer
contributed positively to one output score*.

**What it is not:** a segmentation, a lesion detector, a causal statement, or proof that the model
uses a biological mechanism.

### Health-research perspective

Attribution maps are increasingly included in clinical AI papers as evidence of
"explainability". Treat them as **hypothesis-generating diagnostics of the model**, not as
evidence about the patient or the disease. Two models with identical accuracy can produce very
different maps, and a map that overlaps the lungs is not evidence that the model measured lung
pathology - only that positions inside the lung field influenced the score.

### Think before running

1. Where do you expect the highest Grad-CAM values for a correctly classified pneumonia case?
2. If you compute Grad-CAM for the **normal** logit on the *same* image, do you expect the map to
   be the exact inverse of the pneumonia map?

In [ ]:
class GradCAM:
    # Transparent Grad-CAM in three steps:
    #   1. a forward hook keeps the activations A of the target layer inside the autograd graph;
    #   2. autograd gives us d(score of the chosen class) / dA;
    #   3. the spatially averaged gradient weights the activation maps.
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.handles = [target_layer.register_forward_hook(self._save_activation)]

    def _save_activation(self, module, inputs, output):
        self.activations = output          # NOT detached: we need it in the graph

    def __call__(self, x, class_idx=None):
        # x: (N, C, H, W) prepared input. class_idx: int, array of ints, or None (= argmax).
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        x = x.clone().requires_grad_(True)      # guarantees gradients reach the target layer
        logits = self.model(x)
        if class_idx is None:
            targets = logits.argmax(dim=1)
        elif isinstance(class_idx, int):
            targets = torch.full((x.shape[0],), class_idx, dtype=torch.long, device=x.device)
        else:
            targets = torch.as_tensor(class_idx, dtype=torch.long, device=x.device)
        score = logits.gather(1, targets.view(-1, 1)).sum()

        self.gradients = torch.autograd.grad(score, self.activations)[0].detach()
        self.activations = self.activations.detach()

        alpha = self.gradients.mean(dim=(2, 3), keepdim=True)                # (N, K, 1, 1)
        cam = F.relu((alpha * self.activations).sum(dim=1, keepdim=True))     # (N, 1, h, w)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        cam = cam - cam.amin(dim=(1, 2), keepdim=True)
        cam = cam / (cam.amax(dim=(1, 2), keepdim=True) + 1e-8)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        return cam.detach().cpu().numpy(), probs

    def close(self):
        for h in self.handles:
            h.remove()


def overlay_cam(ax, img_u8, cam, alpha=0.45, title=None):
    ax.imshow(img_u8, cmap="gray", vmin=0, vmax=255)
    ax.imshow(cam, cmap="jet", alpha=alpha, vmin=0, vmax=1)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=8)


print("GradCAM ready. Raw activation shape and gradient shape will match the target layer.")

In [ ]:
# Grad-CAM for six test images, using block3 of Model B as the target layer
cam_engine = GradCAM(cnn, cnn.block3)

sel = np.concatenate([np.flatnonzero(y_te == 1)[:3], np.flatnonzero(y_te == 0)[:3]])
cams, probs_sel = cam_engine(prep_gray(X_te[sel]))            # class = predicted class
print("Grad-CAM output:", cams.shape, "| activations:", tuple(cam_engine.activations.shape),
      "| gradients:", tuple(cam_engine.gradients.shape))

fig, axes = plt.subplots(2, len(sel), figsize=(1.6 * len(sel), 3.8))
for col, i in enumerate(sel):
    axes[0, col].imshow(X_te[i], cmap="gray")
    axes[0, col].axis("off")
    axes[0, col].set_title(f"{CLASS_NAMES[y_te[i]]}\nP(pneu)={probs_sel[col, 1]:.2f}", fontsize=8)
    overlay_cam(axes[1, col], X_te[i], cams[col])
fig.suptitle("Grad-CAM (block3) for the predicted class - top: image, bottom: overlay", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# (a) same image, the two different output logits; (b) same image, different target layers
i = int(sel[0])
x_i = prep_gray(X_te[i])

cam_pneu, p_i = cam_engine(x_i, class_idx=1)
cam_norm, _ = cam_engine(x_i, class_idx=0)

cam_early_engine = GradCAM(cnn, cnn.block1)                   # 32x32 grid
cam_early, _ = cam_early_engine(x_i, class_idx=1)
cam_early_engine.close()

fig, axes = plt.subplots(1, 4, figsize=(9.5, 2.6))
axes[0].imshow(X_te[i], cmap="gray"); axes[0].axis("off")
axes[0].set_title(f"input\ntrue {CLASS_NAMES[y_te[i]]}, P(pneu)={p_i[0, 1]:.2f}", fontsize=8)
overlay_cam(axes[1], X_te[i], cam_pneu[0], title="target = pneumonia logit\n(block3, 16x16)")
overlay_cam(axes[2], X_te[i], cam_norm[0], title="target = normal logit\n(block3, 16x16)")
overlay_cam(axes[3], X_te[i], cam_early[0], title="target = pneumonia logit\n(block1, 32x32)")
plt.tight_layout(); plt.show()

print("correlation between the two class maps: "
      f"{np.corrcoef(cam_pneu[0].ravel(), cam_norm[0].ravel())[0, 1]:.2f}")

# Why we do not use ResNet-18's last block for localisation at this input size
resnet_cam_engine = GradCAM(resnet, resnet.layer4)
cam_rn, _ = resnet_cam_engine(prep_rgb_imagenet(X_te[i]))
resnet_cam_engine.close()
fig, axes = plt.subplots(1, 2, figsize=(4.6, 2.5))
axes[0].imshow(X_te[i], cmap="gray"); axes[0].axis("off"); axes[0].set_title("input", fontsize=8)
overlay_cam(axes[1], X_te[i], cam_rn[0], title="ResNet-18 layer4 Grad-CAM\n(2x2 grid upsampled)")
plt.tight_layout(); plt.show()
print("A 2x2 map upsampled to 64x64 looks smooth but carries only four numbers: "
      "resolution of the target layer is a design choice you must report.")

### Interpretation

Think about each question and discuss it with your neighbour.

1. The two class maps on the same image are usually *similar*, not complementary. Why? Think
   about what the two logits share (the same convolutional features and the same global average
   pooling).
2. Compare the `block1` and `block3` maps. Which one looks more convincing? Which one is more
   *class-specific*? Explain the trade-off between spatial resolution and semantic level.
3. **Does a plausible Grad-CAM map prove that the model learned the correct biological
   mechanism?** Discuss with your neighbour, and state what additional evidence you would require.

### ADDITIONAL Exercise 2 - Grad-CAM on your own selection (optional coding)

> You do not need to write Python to complete this practical. Skip this cell unless you want extra practice.

1. Find the test image where the model is **most confidently wrong** (highest probability for the
   wrong class).
2. Compute and display its Grad-CAM map for (i) the predicted class and (ii) the true class.
3. Do the same for the image where the model is most confidently *right*.
4. Discuss with your neighbour: does the attribution map help you understand the error?

In [ ]:
# ===== EXERCISE 2 =====
# Available: probs_te (N, 2), y_te, X_te, cam_engine, prep_gray, overlay_cam, CLASS_NAMES

# TODO 1: probability assigned to the WRONG class for every test image
#   hint: p_wrong = probs_te[np.arange(len(y_te)), 1 - y_te]
# TODO 2: i_worst = index of the maximum of p_wrong ; i_best = most confident correct case
# TODO 3: cams, _ = cam_engine(prep_gray(X_te[[i_worst]]), class_idx=int(...))
# TODO 4: plot input / CAM(predicted class) / CAM(true class) for both images

---

# Part 6 - Perturbation: does the highlighted region actually matter?

### Concept

An attribution map is a *claim* about the model. Perturbation analysis **tests** that claim:
if the highlighted region really drives the prediction, then destroying the information in that
region should change the predicted probability more than destroying a comparable region elsewhere.

```
image  ->  Grad-CAM  ->  most influential 16x16 patch
                              |
        occlude / blur / change contrast  ->  predict again  ->  compare probabilities
```

Note the essential **control**: a randomly placed patch of the same size. Without it, any drop in
probability could simply be the effect of adding an unusual grey square to the input.

### Health-research perspective

This is a *within-model* sensitivity analysis, the imaging analogue of removing a covariate from
a regression model and observing the change in fit. It says nothing about the disease, only about
the estimator. Sensitivity to a clinically irrelevant region is nevertheless a strong warning
sign for deployment.

### Think before running

1. Occluding the top-attribution region: do you expect P(pneumonia) to fall for every image?
2. What should happen if you occlude a random region instead?
3. Halving global contrast changes no anatomy at all. Should the prediction change?

In [ ]:
def occlude(img_u8, top, left, size=16, value=None):
    out = img_u8.copy()
    out[top:top + size, left:left + size] = int(img_u8.mean()) if value is None else value
    return out


def blur_region(img_u8, top, left, size=16, kernel=7):
    src = torch.from_numpy(img_u8.astype(np.float32)).view(1, 1, *img_u8.shape)
    weight = torch.ones(1, 1, kernel, kernel) / (kernel * kernel)
    blurred = F.conv2d(src, weight, padding=kernel // 2)[0, 0].numpy()
    out = img_u8.astype(np.float32).copy()
    out[top:top + size, left:left + size] = blurred[top:top + size, left:left + size]
    return np.clip(out, 0, 255).astype(np.uint8)


def change_contrast(img_u8, factor=0.5):
    m = float(img_u8.mean())
    return np.clip((img_u8.astype(np.float32) - m) * factor + m, 0, 255).astype(np.uint8)


def top_cam_region(cam, size=16, stride=4):
    # Location of the 'size x size' window with the largest average Grad-CAM value.
    pooled = F.avg_pool2d(torch.from_numpy(cam).view(1, 1, *cam.shape),
                          kernel_size=size, stride=stride)[0, 0].numpy()
    r, c = np.unravel_index(int(pooled.argmax()), pooled.shape)
    return int(r * stride), int(c * stride)


def p_pneumonia(model, img_u8):
    return float(predict_probs(model, img_u8[None, ...])[0, 1])


print("perturbation helpers defined")

In [ ]:
# One image, four perturbations
confident = np.flatnonzero((y_te == 1) & (probs_te[:, 1] > 0.8))
i = int(confident[0]) if len(confident) else int(probs_te[:, 1].argmax())
img = X_te[i]
cam_i, _ = cam_engine(prep_gray(img), class_idx=1)
top, left = top_cam_region(cam_i[0], size=16)
rand_top, rand_left = int(RNG.integers(0, 49)), int(RNG.integers(0, 49))

variants = {
    "original": img,
    f"occlude CAM peak\n(top={top}, left={left})": occlude(img, top, left, 16),
    "occlude random patch": occlude(img, rand_top, rand_left, 16),
    "blur CAM peak": blur_region(img, top, left, 16, kernel=7),
    "global contrast x0.5": change_contrast(img, 0.5),
}

fig, axes = plt.subplots(2, len(variants), figsize=(2.0 * len(variants), 4.2))
base = p_pneumonia(cnn, img)
deltas = {}
for col, (name, im) in enumerate(variants.items()):
    p = p_pneumonia(cnn, im)
    deltas[name] = p - base
    axes[0, col].imshow(im, cmap="gray", vmin=0, vmax=255); axes[0, col].axis("off")
    axes[0, col].set_title(f"{name}\nP(pneu)={p:.3f}", fontsize=7)
    cam_v, _ = cam_engine(prep_gray(im), class_idx=1)
    overlay_cam(axes[1, col], im, cam_v[0])
fig.suptitle("Perturbation and its effect on the prediction (bottom row: Grad-CAM after "
             "perturbation)", fontsize=10)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 2.4))
names = list(deltas)[1:]
plt.bar(range(len(names)), [deltas[n] for n in names],
        color=["tab:red", "tab:grey", "tab:orange", "tab:purple"])
plt.axhline(0, color="black", lw=0.8)
plt.xticks(range(len(names)), [n.split("\n")[0] for n in names], rotation=20, ha="right")
plt.ylabel("change in P(pneumonia)")
plt.title(f"baseline P(pneumonia) = {base:.3f}")
plt.tight_layout(); plt.show()

In [ ]:
# The same comparison over many images, with a matched random-patch control
n_eval = 60
cand = np.flatnonzero(probs_te[:, 1] > 0.5)[:n_eval]
rng_local = np.random.default_rng(1)
drop_cam, drop_rand = [], []

with timed(f"perturbation experiment on {len(cand)} images"):
    for i in cand:
        img = X_te[i]
        base = p_pneumonia(cnn, img)
        cam_i, _ = cam_engine(prep_gray(img), class_idx=1)
        t, l = top_cam_region(cam_i[0], size=16)
        drop_cam.append(base - p_pneumonia(cnn, occlude(img, t, l, 16)))
        rt, rl = int(rng_local.integers(0, 49)), int(rng_local.integers(0, 49))
        drop_rand.append(base - p_pneumonia(cnn, occlude(img, rt, rl, 16)))

drop_cam, drop_rand = np.array(drop_cam), np.array(drop_rand)
print(f"mean drop, Grad-CAM peak patch : {drop_cam.mean():+.3f}")
print(f"mean drop, random patch        : {drop_rand.mean():+.3f}")
print(f"mean paired difference         : {(drop_cam - drop_rand).mean():+.3f} "
      f"(positive => the CAM region matters more)")
print(f"CAM patch caused the larger drop in {(drop_cam > drop_rand).mean()*100:.0f}% of images")

fig, ax = plt.subplots(1, 2, figsize=(8, 2.8))
ax[0].boxplot([drop_cam, drop_rand])
ax[0].set_xticks([1, 2], ["CAM peak", "random"])
ax[0].axhline(0, color="grey", lw=0.8); ax[0].set_ylabel("drop in P(pneumonia)")
ax[0].set_title("occlusion of a 16x16 patch")
ax[1].scatter(drop_rand, drop_cam, s=12, alpha=0.7)
lim = [min(drop_rand.min(), drop_cam.min()) - 0.02, max(drop_rand.max(), drop_cam.max()) + 0.02]
ax[1].plot(lim, lim, "k--", lw=0.8)
ax[1].set_xlabel("random patch"); ax[1].set_ylabel("CAM peak patch")
ax[1].set_title("paired comparison (points above the line\nfavour the attribution map)")
plt.tight_layout(); plt.show()

### Interpretation

Think about each question and discuss it with your neighbour.

1. Did the model behave as the attribution map predicted? Quote the numbers from the paired
   comparison.
2. For which images did occlusion of the CAM peak *increase* the pneumonia probability? What can
   cause this?
3. Occlusion introduces an artefact that never occurs in real radiographs (a flat grey square).
   Why is this a problem for interpreting the result, and what alternative perturbation would be
   more realistic?
4. Global contrast change moved the probability by how much? What does that imply for images
   acquired with a different exposure protocol?

### ADDITIONAL Exercise 3 - quantify attribution with occlusion (optional coding)

> You do not need to write Python to complete this practical. Skip this cell unless you want extra practice.

Write code that, for **20 test images of your choice**:

1. computes Grad-CAM for the predicted class;
2. finds the top-attribution 16x16 region and occludes it;
3. records `delta = P_after - P_before` for the predicted class;
4. reports the mean and the 10th/90th percentile of `delta`, and plots the two most
   perturbation-sensitive images with their maps.

Then discuss with your neighbour: is a large `delta` evidence that the model is *right*?

In [ ]:
# ===== EXERCISE 3 =====
# Available: cam_engine, prep_gray, top_cam_region, occlude, predict_probs, probs_te, X_te, y_te

# TODO 1: choose 20 indices (e.g. the 20 highest-confidence pneumonia predictions)
# TODO 2: loop over them, compute Grad-CAM, occlude the top region, store the probability change
# TODO 3: print mean and np.percentile(deltas, [10, 90])
# TODO 4: display the two images with the largest absolute change, with CAM overlays

---

# Part 7 - Simulated shortcut-learning experiment

> ## SIMULATED DATA MANIPULATION
>
> Everything in this section uses an **artificial marker that we add ourselves**. It is a 6x6
> bright square in the upper-left corner of the image, present in **95% of pneumonia** images and
> **5% of normal** images of our simulated "Site A". This marker does **not** exist in
> PneumoniaMNIST and must never be described as a property of the real data.
>
> It stands in for the many real annotations that leak label information in clinical archives:
> laterality markers, "PORTABLE" or "AP" tags, ECG leads, support devices, exposure indices burned
> into the pixel data, or systematic differences between the scanner used in the emergency
> department and the one used in outpatient clinics.

### Concept - shortcut learning

A model minimises loss. If an irrelevant feature predicts the label *in the training
distribution* and is easier to extract than the real signal, gradient descent will happily use it.
This is not a bug in the optimiser; it is an identification problem in the data.

We train the *same* architecture as before on marker-contaminated data and evaluate it on three
test sets:

| Test set | Marker rule | What it represents |
|---|---|---|
| **Site A test** | same as training (95% / 5%) | internal validation at the same centre |
| **Clean test** | no markers at all | another hospital that does not use this annotation |
| **Swapped test** | marker on 95% of *normal* images | another hospital whose annotation habit differs |

### Think before running

Think about three numbers with your neighbour before you continue:

1. accuracy of the shortcut model on the **Site A** test set;
2. accuracy of the same model on the **clean** test set;
3. accuracy of the same model on the **swapped** test set.

In [ ]:
MARKER_VALUE = 250


def add_marker(imgs, size=6, offset=3, value=MARKER_VALUE):
    # Simulated annotation: a small bright square in the upper-left corner.
    out = imgs.copy()
    out[..., offset:offset + size, offset:offset + size] = value
    return out


def inject_shortcut(X, y, p_marker_pos=0.95, p_marker_neg=0.05, rng=None):
    # Marker presence is correlated with the label, which creates the shortcut.
    rng = np.random.default_rng(0) if rng is None else rng
    probability = np.where(y == 1, p_marker_pos, p_marker_neg)
    marked = rng.random(len(X)) < probability
    out = X.copy()
    out[marked] = add_marker(X[marked])
    return out, marked


rng_shortcut = np.random.default_rng(7)
X_tr_sc, m_tr = inject_shortcut(X_tr, y_tr, rng=rng_shortcut)
X_va_sc, m_va = inject_shortcut(X_va, y_va, rng=rng_shortcut)
X_te_siteA, m_teA = inject_shortcut(X_te, y_te, rng=rng_shortcut)      # same habit as training
X_te_clean = X_te.copy()                                               # no annotation at all
X_te_swap, m_swap = inject_shortcut(X_te, 1 - y_te, rng=rng_shortcut)  # opposite habit

for name, y, marked in [("train", y_tr, m_tr), ("Site A test", y_te, m_teA),
                        ("swapped test", y_te, m_swap)]:
    print(f"{name:13s}: marker in {100*marked[y == 1].mean():5.1f}% of pneumonia, "
          f"{100*marked[y == 0].mean():5.1f}% of normal images")

fig, axes = plt.subplots(2, 6, figsize=(10, 3.8))
ids = np.concatenate([np.flatnonzero((y_tr == 1) & m_tr)[:3],
                      np.flatnonzero((y_tr == 0) & ~m_tr)[:3]])
for col, i in enumerate(ids):
    axes[0, col].imshow(X_tr_sc[i], cmap="gray", vmin=0, vmax=255)
    axes[0, col].set_title(f"{CLASS_NAMES[y_tr[i]]}\nmarker={bool(m_tr[i])}", fontsize=8)
    axes[1, col].imshow(X_tr_sc[i][:20, :20], cmap="gray", vmin=0, vmax=255)
    axes[1, col].set_title("corner zoom", fontsize=7)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("SIMULATED shortcut: 6x6 bright marker added by us, correlated with the label",
             fontsize=10)
plt.tight_layout(); plt.show()
print(f"\nthe marker occupies {6*6}/{64*64} = {100*36/4096:.1f}% of the pixels")

In [ ]:
# Same architecture on contaminated data. Default: load instructor weights (TRAIN_SHORTCUT=False).
cnn_shortcut = SmallCNN().to(DEVICE)
with timed("obtain SmallCNN weights (shortcut images)"):
    history_sc = load_or_train(
        cnn_shortcut, "cnn_shortcut.pt", TRAIN_SHORTCUT,
        lambda: train_classifier(cnn_shortcut, X_tr_sc, y_tr, X_va_sc, y_va,
                                 epochs=6, batch=64, lr=1e-3),
        "shortcut SmallCNN")

if history_sc is not None:
    plt.figure(figsize=(4, 2.4))
    plt.plot(history_sc["epoch"], history_sc["val_acc"], marker="o", color="tab:red",
             label="shortcut model")
    if history_clean is not None:
        plt.plot(history_clean["epoch"], history_clean["val_acc"], marker="o", color="tab:green",
                 label="clean model")
    plt.xlabel("epoch"); plt.ylabel("validation accuracy (Site A style)")
    plt.ylim(0.5, 1.02); plt.legend(); plt.title("Validation looks excellent")
    plt.tight_layout(); plt.show()

In [ ]:
# Evaluate BOTH models on the three test sets
test_sets = {"Site A test\n(marker as in training)": X_te_siteA,
             "clean test\n(no markers)": X_te_clean,
             "swapped test\n(marker on normals)": X_te_swap}
models = {"shortcut model": cnn_shortcut, "clean model": cnn}

results = {}
print(f"{'model':16s} {'test set':22s} {'accuracy':>9s} {'balanced acc':>13s} {'ROC-AUC':>8s}")
for mname, model in models.items():
    for tname, X_set in test_sets.items():
        p = predict_probs(model, X_set)
        acc = accuracy_score(y_te, p.argmax(1))
        bal = balanced_accuracy_score(y_te, p.argmax(1))
        auc = roc_auc_score(y_te, p[:, 1])
        results[(mname, tname)] = (acc, bal, auc)
        print(f"{mname:16s} {tname.replace(chr(10), ' '):22s} {acc:>9.3f} {bal:>13.3f} {auc:>8.3f}")

labels = list(test_sets)
xpos = np.arange(len(labels))
plt.figure(figsize=(7.5, 3))
for offset, (mname, colour) in zip([-0.18, 0.18],
                                   [("shortcut model", "tab:red"), ("clean model", "tab:green")]):
    plt.bar(xpos + offset, [results[(mname, t)][0] for t in labels], width=0.34,
            label=mname, color=colour)
plt.axhline(0.5, color="grey", ls="--", lw=0.8)
plt.xticks(xpos, labels, fontsize=8)
plt.ylabel("accuracy"); plt.ylim(0, 1.05)
plt.title("The same architecture, two training sets: internal accuracy hides the problem")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Where does the shortcut model look?
cam_sc = GradCAM(cnn_shortcut, cnn_shortcut.block3)
marked_pneu = np.flatnonzero((y_te == 1) & m_teA)[:4]

cams_sc, probs_sc = cam_sc(prep_gray(X_te_siteA[marked_pneu]), class_idx=1)
cams_ok, probs_ok = cam_engine(prep_gray(X_te_siteA[marked_pneu]), class_idx=1)

fig, axes = plt.subplots(3, len(marked_pneu), figsize=(2.0 * len(marked_pneu), 6.0))
for col, i in enumerate(marked_pneu):
    axes[0, col].imshow(X_te_siteA[i], cmap="gray", vmin=0, vmax=255); axes[0, col].axis("off")
    axes[0, col].set_title("input (with marker)", fontsize=8)
    overlay_cam(axes[1, col], X_te_siteA[i], cams_sc[col],
                title=f"shortcut model\nP(pneu)={probs_sc[col, 1]:.2f}")
    overlay_cam(axes[2, col], X_te_siteA[i], cams_ok[col],
                title=f"clean model\nP(pneu)={probs_ok[col, 1]:.2f}")
fig.suptitle("Grad-CAM exposes the shortcut: attention on the corner annotation", fontsize=10)
plt.tight_layout(); plt.show()
cam_sc.close()

### Interpretation - the questions that matter

Think about each question and discuss it with your neighbour.

1. **Is this a good model?** Internal validation accuracy was very high. State precisely what the
   internal estimate is an unbiased estimate *of*.
2. **Would this performance survive deployment at another hospital** that does not use the same
   annotation? Which of the three test sets is the closest analogue of that situation?
3. **How does this resemble confounding?** Formulate it in epidemiological language: what is the
   exposure, the outcome, and the confounder? Why does the model's use of the marker resemble an
   unadjusted association?
4. **What kind of external validation would detect the problem?** Consider: geographic external
   validation, temporal validation, subgroup analysis by scanner, prospective evaluation, and
   inspection of attribution maps. Rank them by how convincingly they would reveal this failure.
5. Grad-CAM revealed the shortcut here because the shortcut was **spatially localised and
   visible**. Give an example of a shortcut that Grad-CAM would *not* reveal.

### ADDITIONAL Exercise 4 - identify and quantify the shortcut (optional coding)

> You do not need to write Python to complete this practical. Skip this cell unless you want extra practice.

You are handed the shortcut model without being told what is wrong with it. Design and run a
diagnostic:

1. Take the pneumonia test images that carry a marker. **Remove** the marker (use the original
   `X_te`) and record the change in P(pneumonia) for the shortcut model *and* for the clean model.
2. Take normal test images without a marker, **add** one, and record the same quantities.
3. Report the mean change per model in a small table.
4. Discuss with your neighbour, as if it were the limitations section of a paper:
   what you found, what the mechanism is, why internal validation missed it, and what this implies
   for **external validity and transportability** of the reported accuracy.

In [ ]:
# ===== EXERCISE 4 =====
# Available: X_te, y_te, m_teA, X_te_siteA, add_marker, predict_probs, cnn, cnn_shortcut

# TODO 1: idx_marked_pneu = pneumonia test images that carry a marker in X_te_siteA
# TODO 2: for both models, compute mean P(pneumonia) with the marker (X_te_siteA[idx])
#         and without it (X_te[idx]); store the difference
# TODO 3: idx_clean_normal = normal test images WITHOUT a marker; compare X_te[idx]
#         with add_marker(X_te[idx])
# TODO 4: print a table: rows = model, columns = 'marker removed', 'marker added'
# TODO 5: discuss the limitations with your neighbour, as if it were the limitations section of a paper

---

# Wrap-up

### What we did

```
pretrained CNN + small task-trained CNN
        -> inspect architecture, parameters, tensor shapes
        -> read feature maps at increasing depth      (representation)
        -> Grad-CAM for a chosen output               (attribution)
        -> perturb the highlighted region             (test the attribution claim)
        -> contaminate the data with a fake marker    (shortcut learning)
        -> discover that internal validation cannot detect it
```

### Reflection questions

Discuss these with your neighbour, then in the plenary.

1. Accuracy, calibration and attribution maps are three different things. For a model intended to
   triage chest radiographs, which of the three would you require in a manuscript, and which would
   you require before clinical deployment?
2. Grad-CAM is *class-specific but not mechanism-specific*. Design an experiment (data, not
   visualisation) that would provide evidence that a model uses lung parenchyma rather than
   patient positioning.
3. A colleague reports 0.97 AUC for pneumonia detection developed and validated at a single
   centre with a random split. List the three most likely explanations that are not
   "the model detects pneumonia".
4. Shortcut learning and confounding are the same statistical phenomenon in different vocabularies.
   Where does the analogy break down? Consider adjustment: can you "adjust" a CNN for a confounder?
5. Which single change to the *study design* (not the model) would most improve the credibility of
   a medical imaging model: multi-centre training data, prospective evaluation, external test set
   from a different scanner vendor, or an attribution-map figure? Defend your ranking.

6. **Suppose internal AUC = 0.94 and external AUC = 0.71.** Grad-CAM also changes substantially between hospitals. What hypotheses would you investigate **before** retraining the model? Consider scanner/protocol, prevalence and case mix, acquisition artefacts, demographic composition, label definition, selection into the imaging pathway, calibration, and shortcut features.

### Methods you can reuse

| Task | Tool used here |
|---|---|
| feature-map inspection | `register_forward_hook` |
| attribution | Grad-CAM implemented in ~30 lines |
| attribution testing | occlusion with a matched random-patch control |
| shortcut diagnosis | train/evaluate under a *modified* marker distribution |

### Key references

* Selvaraju, R. R. et al. (2017). Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization. *ICCV*.
* Geirhos, R. et al. (2020). Shortcut learning in deep neural networks. *Nature Machine Intelligence*, 2, 665-673.
* Zech, J. R. et al. (2018). Variable generalization performance of a deep learning model to detect pneumonia in chest radiographs: a cross-sectional study. *PLOS Medicine*, 15(11), e1002683.
* DeGrave, A. J., Janizek, J. D., Lee, S.-I. (2021). AI for radiographic COVID-19 detection selects shortcuts over signal. *Nature Machine Intelligence*, 3, 610-619.
* Adebayo, J. et al. (2018). Sanity checks for saliency maps. *NeurIPS*.
* Yang, J. et al. (2023). MedMNIST v2. *Scientific Data*, 10, 41.

*Model solutions for the additional coding exercises are in `README_Instructor.md`.*